# YOLOv8n baselines for jersey-number localisation

Companion to `kaggle_train.ipynb` in
[MHKhan50/jersey-number-detection](https://github.com/MHKhan50/jersey-number-detection).
It trains YOLOv8n in up to four settings and scores every run with **the same evaluator as
JerseyNet** (COCO-style AP over all boxes, first-box IoU, 95% bootstrap intervals):

| Setting | Weights | Input | Epochs | What it isolates |
|---|---|---|---|---|
| COCO-pretrained, 640 px | `yolov8n.pt` | 640 | 50 | the earlier comparison |
| COCO-pretrained, 256 px | `yolov8n.pt` | 256 | 50 | pretraining, at JerseyNet's input size |
| From scratch, 256 px | `yolov8n.yaml` | 256 | 140 | architecture only, no pretraining |
| From scratch, 384 px | `yolov8n.yaml` | 384 | 140 | matches JerseyNet's best recipe (384 px) |

Each setting runs with seeds 0, 1 and 2 (all settings for seed 0 first). `ONLY` in the
settings cell limits the run to some settings; by default it runs only the 384 px from-scratch
setting, as the other three are already reported in the repository. Checkpoints are chosen
on validation; the test split is only used for the final scores.

**Kaggle settings:** Accelerator *GPU T4 x2*, Internet *on* (Ultralytics and the COCO weights
are downloaded), the `jersey-2` dataset attached under *Input*. Then *Save Version → Save & Run
All (Commit)*. Results go to `/kaggle/working/yolo_results`. Re-running with the previous
version's output attached as an input is not needed: unfinished runs simply start again.

In [ ]:
# Settings
import os

REPO_URL = "https://github.com/MHKhan50/jersey-number-detection.git"
REPO_REF = "main"
TIME_BUDGET_HOURS = 9.5          # no new run starts after this; Kaggle stops at 12 h
SEEDS = [0, 1, 2]
ONLY = ["yolov8n_from_scratch_384_px"]   # [] runs all four settings
OUT_DIR = "/kaggle/working/yolo_results"
INPUT_ROOTS = ["/kaggle/input"]

# Continuous-integration smoke test only: synthetic data, one tiny CPU run.
SMOKE = os.environ.get("JND_SMOKE") == "1"
if SMOKE:
    SEEDS, OUT_DIR = [0], os.environ.get("JND_OUT", "/tmp/jnd-yolo-smoke/results")
print({"REPO_REF": REPO_REF, "SEEDS": SEEDS, "TIME_BUDGET_HOURS": TIME_BUDGET_HOURS, "SMOKE": SMOKE})

In [ ]:
# Environment check and Ultralytics (PyTorch from the Kaggle image is kept as it is)
import subprocess
import sys

import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif not SMOKE:
    print("WARNING: no GPU. Set Accelerator to GPU T4 x2.")
try:
    import ultralytics  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3"], check=True)
import ultralytics

print("ultralytics", ultralytics.__version__)

In [ ]:
# Get the code
import time
from pathlib import Path

REPO_DIR = Path(os.environ.get("JND_REPO_DIR", "/tmp/jersey-number-detection"))
if not (REPO_DIR / "src" / "jersey_detect").exists():
    for attempt in range(3):
        if subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)]).returncode == 0:
            break
        time.sleep(10)
SRC = str(REPO_DIR / "src")
sys.path.insert(0, SRC)
print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Find the dataset
from jersey_detect.prepare import find_dataset

if SMOKE:
    from jersey_detect.synthetic import make_dataset
    DATA_ROOT = make_dataset("/tmp/jnd-yolo-smoke/data")
else:
    DATA_ROOT = find_dataset(INPUT_ROOTS)
if DATA_ROOT is None:
    raise RuntimeError("No dataset: attach jersey-2 under Input (folder with train/, valid/, test/)")
print("Dataset:", DATA_ROOT)

In [ ]:
# Train and score every setting in a separate process (a crash cannot take the notebook down)
env = dict(os.environ, PYTHONPATH=SRC, PYTHONUNBUFFERED="1")
cmd = [sys.executable, "-m", "jersey_detect", "yolo", "--data", str(DATA_ROOT), "--out", OUT_DIR,
       "--hours", str(TIME_BUDGET_HOURS), "--seeds", *map(str, SEEDS)]
if ONLY and not SMOKE:
    cmd += ["--only", *ONLY]
if SMOKE:
    cmd.append("--smoke")
print(" ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
print("return code:", proc.wait())

In [ ]:
# Results
import json

summary_path = Path(OUT_DIR) / "summary.json"
if summary_path.exists():
    print((Path(OUT_DIR) / "results.md").read_text())
    summary = json.loads(summary_path.read_text())
    if summary["errors"]:
        print("Errors:", json.dumps(summary["errors"], indent=1))
    if summary.get("skipped"):
        print("Skipped (time budget):", summary["skipped"])
else:
    print("No summary written; see the output above.")